# Intro to MLOps — the Model Registry (notebook 3)

Notebooks 1 and 2 *tracked* experiments. This one *manages* the resulting models
with the **MLflow Model Registry** — the standard way to answer "which model is
in production, what version, and how do I load it?"

- Tutorial: <https://jienweng.github.io/notes/intro-to-mlops-tutorial/>

**Registry vocabulary**
- **Registered model** — a named slot, e.g. `churn-classifier`.
- **Version** — every time you register a model it gets an incrementing number (v1, v2, …).
- **Alias** — a movable pointer like `@champion` / `@challenger`. The modern
  replacement for the deprecated *Stages* — you load `models:/name@champion` and
  swap which version it points at without touching your serving code.
- **Tags / description** — metadata that travels with the model (e.g. `validation=passed`).

Run notebooks 1 and 2 first so there are runs to promote. This notebook only
reads their runs and registers the best of each — it logs no new training runs.

## Setup

In [1]:
import pandas as pd
import mlflow
from mlflow import MlflowClient

mlflow.set_tracking_uri('sqlite:///mlflow.db')
client = MlflowClient()

# Each experiment -> the registered-model name and the metric we rank runs by.
REGISTRY = [
    ('intro-to-mlops',   'breast-cancer-classifier', 'accuracy'),
    ('churn-prediction', 'churn-classifier',          'test_accuracy'),
]

/Users/jienweng/miniforge3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Find the best run in each experiment

The registry promotes *runs*, so first pick the best run per experiment by its
metric. Autolog and manual logging both store the model at `runs:/<id>/model`.

In [2]:
def top_runs(experiment_name, metric, n=2):
    """Return up to n runs (best first) that have the given metric."""
    exp = client.get_experiment_by_name(experiment_name)
    if exp is None:
        return []
    return client.search_runs(
        [exp.experiment_id],
        filter_string=f'metrics.{metric} > 0',
        order_by=[f'metrics.{metric} DESC'],
        max_results=n)

for exp_name, model_name, metric in REGISTRY:
    runs = top_runs(exp_name, metric, n=1)
    if runs:
        r = runs[0]
        print(f'{exp_name:18s} best {metric}={r.data.metrics[metric]:.4f}  run={r.info.run_id[:8]}')
    else:
        print(f'{exp_name:18s} no runs yet — run that notebook first')

intro-to-mlops     best accuracy=0.9649  run=3054e997
churn-prediction   best test_accuracy=0.7550  run=acf60bd8


## 2. Register the best model of each experiment

`register_model` copies the run's model into the named slot and returns its new
version number. Registering the same name again just adds another version.

In [3]:
registered = {}   # model_name -> version

for exp_name, model_name, metric in REGISTRY:
    runs = top_runs(exp_name, metric, n=1)
    if not runs:
        print(f'skip {model_name}: no runs in {exp_name!r}')
        continue
    best = runs[0]
    mv = mlflow.register_model(f'runs:/{best.info.run_id}/model', model_name)
    registered[model_name] = int(mv.version)
    print(f'{model_name}: registered v{mv.version} '
          f'({metric}={best.data.metrics[metric]:.4f})')

Registered model 'breast-cancer-classifier' already exists. Creating a new version of this model...
2026/07/01 17:29:12 WARNING mlflow.tracking._model_registry.fluent: Run with id 3054e997571a47068a57c311693a7d61 has no artifacts at artifact path 'model', registering model based on models:/m-86da6c5422e041bcab00b577c6a9b7c6 instead


Created version '3' of model 'breast-cancer-classifier'.
Registered model 'churn-classifier' already exists. Creating a new version of this model...
2026/07/01 17:29:12 WARNING mlflow.tracking._model_registry.fluent: Run with id acf60bd864c04d3daf986fa46c77d13b has no artifacts at artifact path 'model', registering model based on models:/m-9712ad7343ca4b4caa43154c29dd2939 instead


breast-cancer-classifier: registered v3 (accuracy=0.9649)
churn-classifier: registered v2 (test_accuracy=0.7550)


Created version '2' of model 'churn-classifier'.


## 3. Add metadata, then alias the production model — `@champion`

Description and tags document the model; the `champion` alias marks the version
you consider production. Serving code then loads `models:/<name>@champion` and
never has to hard-code a version number.

In [4]:
for model_name, version in registered.items():
    client.update_registered_model(
        model_name, description=f'Best tracked model for {model_name}.')
    client.set_registered_model_tag(model_name, 'task', 'classification')
    client.set_model_version_tag(model_name, version, 'validation', 'passed')
    client.set_registered_model_alias(model_name, 'champion', version)
    print(f'{model_name} v{version}  ->  @champion  (tagged validation=passed)')

breast-cancer-classifier v3  ->  @champion  (tagged validation=passed)
churn-classifier v2  ->  @champion  (tagged validation=passed)


## 4. Load a model *by alias* and predict

This is the payoff: no run IDs, no version numbers in your code — just the name
and the alias.

In [5]:
if 'churn-classifier' in registered:
    model = mlflow.sklearn.load_model('models:/churn-classifier@champion')
    churn = pd.read_csv('churn.csv')
    X = pd.get_dummies(churn.drop(columns='churn'), drop_first=True)
    print('Loaded models:/churn-classifier@champion')
    print('Sample predictions (1 = churn):', model.predict(X.head()).tolist())
else:
    print('Run notebook 2 first to register churn-classifier.')

Loaded models:/churn-classifier@champion
Sample predictions (1 = churn): [0, 0, 0, 0, 0]


## 5. Version management — a challenger, then a swap

New model → register as a new version and alias it `@challenger`. Evaluate it
offline; when it wins, point `@champion` at it. Nothing else in your stack
changes — the alias moves, the code stays the same.

In [6]:
name = next(iter(registered), None)
if name is None:
    print('Nothing registered yet.')
else:
    exp_name, metric = next((e, m) for e, n, m in REGISTRY if n == name)
    runs = top_runs(exp_name, metric, n=2)
    # second-best run if available, else re-register the best (demo: creates a new version)
    source = runs[1] if len(runs) > 1 else runs[0]
    challenger = mlflow.register_model(f'runs:/{source.info.run_id}/model', name)
    client.set_registered_model_alias(name, 'challenger', int(challenger.version))
    print(f'{name} v{challenger.version}  ->  @challenger')

    # ...decide the challenger wins, so promote it:
    client.set_registered_model_alias(name, 'champion', int(challenger.version))
    print(f'promoted {name} v{challenger.version} to @champion '
          f'(was v{registered[name]})')

Registered model 'breast-cancer-classifier' already exists. Creating a new version of this model...
2026/07/01 17:29:13 WARNING mlflow.tracking._model_registry.fluent: Run with id d62233ac64f04769b2dcd9fe38ae1090 has no artifacts at artifact path 'model', registering model based on models:/m-76585e4d0efe4765aee9436d1dfb4a59 instead


breast-cancer-classifier v4  ->  @challenger
promoted breast-cancer-classifier v4 to @champion (was v3)


Created version '4' of model 'breast-cancer-classifier'.


## 6. Browse the registry

In [7]:
for rm in client.search_registered_models():
    print(f'{rm.name}   aliases={rm.aliases}')
    for v in client.search_model_versions(f"name='{rm.name}'"):
        print(f'   v{v.version}  run={v.run_id[:8]}  tags={dict(v.tags)}')

Breast_Cancer_Prediction   aliases={}
   v1  run=05268b11  tags={}
breast-cancer-classifier   aliases={'challenger': 4, 'champion': 4}
   v4  run=d62233ac  tags={}
   v3  run=3054e997  tags={'validation': 'passed'}
   v2  run=d62233ac  tags={}
   v1  run=3054e997  tags={'validation': 'passed'}
churn-classifier   aliases={'champion': 2}
   v2  run=acf60bd8  tags={'validation': 'passed'}
   v1  run=a5a671ce  tags={'validation': 'passed'}


## Recap

- **Track** with notebooks 1–2, then **register** the best run's model here.
- Load with `models:/<name>@champion` — versions and aliases keep serving code stable.
- Promote by *moving an alias*, not by editing code or copying files.

Everything lives in the same local `mlflow.db`; view it all under **Models** in
`mlflow ui --backend-store-uri sqlite:///mlflow.db`.